# Amazon ML Challenge: Stage 2 — Normalization, Multilingual Keys & Ground-Truth Blocking Benchmark

This notebook implements the complete pipeline for Stage 2:
1. **Configuring Paths & Verifying Existing Directories**
2. **DuckDB Initialization (Multi-threaded & Memory-budgeted)**
3. **Advanced Normalization (Unicode -> Transliteration -> Legal Suffix Removal -> Blocking Keys)**
4. **Interactive Validation on Multilingual Test Records**
5. **Streaming Generation of Compact Parquet Key Files (`s1_keys.parquet`, `s2_keys.parquet`, `s3_keys.parquet`)**
6. **Key File Verification**
7. **Exploding Ground Truth into S1->S2/S3 Pair Parquet (`gt_pairs.parquet`)**
8. **Ground-Truth Country Consistency Analysis**
9. **Block Size Distribution Metrics (Mean, Median, P90, P95, P99, Max)**
10. **Theoretical Candidate Pair Estimation (Exact Cartesian products without materialization)**
11. **Empirical Ground-Truth Pair Recall Evaluation**
12. **Full Entity Recall Evaluation (100% complete match recovery)**
13. **Comprehensive Recall vs. Cost Benchmark Table**
14. **Country-Specific Name Uniqueness Analysis**
15. **Matched Cross-Script Distribution Analysis**
16. **Sampling Known True Pairs for Feature / Similarity Exploration**
17. **In-Memory Candidate Generation & Ground-Truth Recall Benchmark**


## 1. Setup, Imports & Paths

In [1]:
from pathlib import Path
import os
import sys
import re
import gc
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from unidecode import unidecode
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Import modular helper functions
sys.path.append(os.path.abspath(".."))
import src.blocking_utils as bu

# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path("../student_resource/dataset/train")

S1_PATH = BASE_DIR / "train_source1.tsv"
S2_PATH = BASE_DIR / "train_source2.tsv"
S3_PATH = BASE_DIR / "train_source3.tsv"
GT_PATH = BASE_DIR / "train_ground_truth.tsv"

OUTPUT_DIR = Path("../outputs")
BLOCK_DIR = OUTPUT_DIR / "blocking_keys"
BLOCK_DIR.mkdir(parents=True, exist_ok=True)

# Performance tuning
CHUNK_SIZE = 250_000

print("S1:", S1_PATH.resolve(), "(exists:", S1_PATH.exists(), ")")
print("S2:", S2_PATH.resolve(), "(exists:", S2_PATH.exists(), ")")
print("S3:", S3_PATH.resolve(), "(exists:", S3_PATH.exists(), ")")
print("GT:", GT_PATH.resolve(), "(exists:", GT_PATH.exists(), ")")
print("\nBlocking output:", BLOCK_DIR.resolve())


S1: D:\ml-model\amazon-ml\student_resource\dataset\train\train_source1.tsv (exists: True )
S2: D:\ml-model\amazon-ml\student_resource\dataset\train\train_source2.tsv (exists: True )
S3: D:\ml-model\amazon-ml\student_resource\dataset\train\train_source3.tsv (exists: True )
GT: D:\ml-model\amazon-ml\student_resource\dataset\train\train_ground_truth.tsv (exists: True )

Blocking output: D:\ml-model\amazon-ml\outputs\blocking_keys


## 2. Initialize DuckDB Engine

In [2]:
con = duckdb.connect()

con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='12GB';")

print("DuckDB Version:", con.execute("SELECT version();").fetchone()[0])


DuckDB Version: v1.4.0


## 3. Verify Normalization & Transliteration on Multilingual Examples

In [3]:
test_df = pd.DataFrame({
    "entity_id": [
        "TEST-1",
        "TEST-2",
        "TEST-3",
        "TEST-4",
    ],
    "business_name": [
        "Acme Robotics Pvt Ltd",
        "इंडो फूड",
        "Ambika Ienal Limited",
        "Dermatology Global Specialists LLC",
    ],
    "business_address": [
        "42 MG Rd, Bengaluru 560001",
        "PLOT 71 D/2, OMKARESHWAR SOCIETY, BORIVALI WEST, MUMBAI",
        "No. 126, 1 St Stage, 3Rd Block, Nagarbhavi, ಕರ್ನಾಟಕ",
        "Omaha, NE, 2214 Grand Avenue",
    ],
    "country": [
        "India",
        "India",
        "India",
        "US",
    ]
})

test_features = bu.build_block_features(test_df)

display(
    test_features[
        [
            "entity_id",
            "name_script",
            "address_script",
            "name_translit",
            "name_expanded",
            "name_core",
            "name_prefix4",
            "name_prefix6",
            "name_token_pair",
            "postal_code",
            "block_postal_name4",
            "block_name_prefix6",
            "block_name_exact",
            "block_token_pair",
        ]
    ]
)


,entity_id,name_script,address_script,name_translit,name_expanded,name_core,name_prefix4,name_prefix6,name_token_pair,postal_code,block_postal_name4,block_name_prefix6,block_name_exact,block_token_pair
0,TEST-1,LATIN,LATIN,acme robotics pvt ltd,acme robotics private limited,acme robotics,acme,acmero,acme|robotics,560001,INDIA|560001|acme,INDIA|acmero,INDIA|acme robotics,INDIA|acme|robotics
1,TEST-2,DEVANAGARI,LATIN,inddo phuudd,inddo phuudd,inddo phuudd,indd,inddop,inddo|phuudd,,INDIA||indd,INDIA|inddop,INDIA|inddo phuudd,INDIA|inddo|phuudd
2,TEST-3,LATIN,MIXED,ambika ienal limited,ambika ienal limited,ambika ienal,ambi,ambika,ambika|ienal,,INDIA||ambi,INDIA|ambika,INDIA|ambika ienal,INDIA|ambika|ienal
3,TEST-4,LATIN,LATIN,dermatology global specialists llc,dermatology global specialists llc,dermatology global specialists,derm,dermat,dermatology|specialists,,US||derm,US|dermat,US|dermatology global specialists,US|dermatology|specialists


## 4. Generate Compact Blocking-Key Parquet Files

In [4]:
S1_KEYS = BLOCK_DIR / "s1_keys.parquet"
S2_KEYS = BLOCK_DIR / "s2_keys.parquet"
S3_KEYS = BLOCK_DIR / "s3_keys.parquet"

print("Building S1 Keys Parquet...")
bu.write_key_parquet(S1_PATH, S1_KEYS, CHUNK_SIZE)

print("Building S2 Keys Parquet...")
bu.write_key_parquet(S2_PATH, S2_KEYS, CHUNK_SIZE)

print("Building S3 Keys Parquet...")
bu.write_key_parquet(S3_PATH, S3_KEYS, CHUNK_SIZE)


Building S1 Keys Parquet...


Building s1_keys.parquet: 0it [00:00, ?it/s]

Written 2,206,821 rows -> ..\outputs\blocking_keys\s1_keys.parquet
Building S2 Keys Parquet...


Building s2_keys.parquet: 0it [00:00, ?it/s]

Written 5,034,616 rows -> ..\outputs\blocking_keys\s2_keys.parquet
Building S3 Keys Parquet...


Building s3_keys.parquet: 0it [00:00, ?it/s]

Written 5,285,603 rows -> ..\outputs\blocking_keys\s3_keys.parquet


## 5. Verify Generated Key Files

In [5]:
for path in [S1_KEYS, S2_KEYS, S3_KEYS]:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT entity_id) AS unique_ids
        FROM read_parquet('{path.resolve().as_posix()}')
    """).df()
    print("\n", path.name)
    display(result)



 s1_keys.parquet


,rows,unique_ids
0,2206821,2206821



 s2_keys.parquet


,rows,unique_ids
0,5034616,5034616



 s3_keys.parquet


,rows,unique_ids
0,5285603,5285603


## 6. Convert Ground Truth into Individual S1->S2/S3 Pairs

In [6]:
GT_PAIRS = OUTPUT_DIR / "gt_pairs.parquet"

if GT_PAIRS.exists():
    GT_PAIRS.unlink()

gt_sql_path = GT_PATH.resolve().as_posix()

query = f"""
COPY (
    SELECT
        source1_entity_id,
        TRIM(matched_entity_id) AS matched_entity_id,
        CASE
            WHEN TRIM(matched_entity_id) LIKE 'S2-%' THEN 'S2'
            WHEN TRIM(matched_entity_id) LIKE 'S3-%' THEN 'S3'
            ELSE 'UNKNOWN'
        END AS target_source
    FROM read_csv(
        '{gt_sql_path}',
        delim='\t',
        header=true,
        columns={{
            'source1_entity_id': 'VARCHAR',
            'matched_entity_ids': 'VARCHAR'
        }}
    )
    CROSS JOIN UNNEST(
        STRING_SPLIT(
            COALESCE(matched_entity_ids, ''),
            ','
        )
    ) AS t(matched_entity_id)
    WHERE TRIM(matched_entity_id) <> ''
) TO '{GT_PAIRS.resolve().as_posix()}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
"""

con.execute(query)

gt_pair_count = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{GT_PAIRS.resolve().as_posix()}')
""").fetchone()[0]

print(f"Total positive S1->S2/S3 links in Ground Truth: {gt_pair_count:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total positive S1->S2/S3 links in Ground Truth: 7,638,365


## 7. Ground Truth Country Consistency Analysis

In [7]:
def country_consistency_analysis():
    queries = []
    for target_name, target_path in [("S2", S2_KEYS), ("S3", S3_KEYS)]:
        q = f"""
        SELECT
            '{target_name}' AS target_source,
            COUNT(*) AS total_pairs,
            SUM(
                CASE
                    WHEN s1.country <> tgt.country THEN 1
                    ELSE 0
                END
            ) AS country_mismatch_pairs,
            100.0 * SUM(
                CASE
                    WHEN s1.country <> tgt.country THEN 1
                    ELSE 0
                END
            ) / COUNT(*) AS mismatch_percent
        FROM read_parquet('{GT_PAIRS.resolve().as_posix()}') g
        INNER JOIN read_parquet('{S1_KEYS.resolve().as_posix()}') s1
            ON g.source1_entity_id = s1.entity_id
        INNER JOIN read_parquet('{target_path.resolve().as_posix()}') tgt
            ON g.matched_entity_id = tgt.entity_id
        WHERE g.target_source = '{target_name}'
        """
        queries.append(q)

    result = con.execute(queries[0] + "\nUNION ALL\n" + queries[1]).df()
    return result

country_result = country_consistency_analysis()
display(country_result)


,target_source,total_pairs,country_mismatch_pairs,mismatch_percent
0,S2,3693619,0.0,0.0
1,S3,3944746,0.0,0.0


## 8. Measure Block Size Distributions

In [8]:
BLOCK_RULES = [
    "block_postal_name4",
    "block_name_prefix6",
    "block_name_exact",
    "block_token_pair",
    "block_postal",
]

def block_size_stats(parquet_path, rule):
    path = parquet_path.resolve().as_posix()
    query = f"""
    WITH blocks AS (
        SELECT
            "{rule}" AS block_key,
            COUNT(*) AS cnt
        FROM read_parquet('{path}')
        WHERE "{rule}" <> ''
        GROUP BY "{rule}"
    )
    SELECT
        COUNT(*) AS number_of_blocks,
        SUM(cnt) AS rows_in_nonempty_blocks,
        AVG(cnt) AS mean_block_size,
        MEDIAN(cnt) AS median_block_size,
        quantile_cont(cnt, 0.90) AS p90_block_size,
        quantile_cont(cnt, 0.95) AS p95_block_size,
        quantile_cont(cnt, 0.99) AS p99_block_size,
        MAX(cnt) AS max_block_size
    FROM blocks;
    """
    result = con.execute(query).df()
    result.insert(0, "rule", rule)
    result.insert(1, "source", parquet_path.stem)
    return result

block_stats_all = []
for path in [S1_KEYS, S2_KEYS, S3_KEYS]:
    for rule in BLOCK_RULES:
        result = block_size_stats(path, rule)
        block_stats_all.append(result)

block_stats = pd.concat(block_stats_all, ignore_index=True)
display(block_stats.sort_values(["rule", "source"]))


,rule,source,number_of_blocks,rows_in_nonempty_blocks,mean_block_size,median_block_size,p90_block_size,p95_block_size,p99_block_size,max_block_size
2,block_name_exact,s1_keys,1374502,2206821.0,1.605542,1.0,2.0,3.0,7.0,523
7,block_name_exact,s2_keys,3474832,5034613.0,1.448880,1.0,2.0,3.0,7.0,695
12,block_name_exact,s3_keys,3717502,5285590.0,1.421812,1.0,2.0,3.0,7.0,622
1,block_name_prefix6,s1_keys,339448,2206821.0,6.501205,1.0,6.0,13.0,69.0,15070
6,block_name_prefix6,s2_keys,563975,5034613.0,8.927014,1.0,8.0,18.0,99.0,54651
11,block_name_prefix6,s3_keys,573437,5285590.0,9.217386,1.0,9.0,20.0,102.0,63871
4,block_postal,s1_keys,30812,145101.0,4.709237,2.0,12.0,18.0,33.0,87
9,block_postal,s2_keys,42664,315765.0,7.401205,3.0,19.0,29.0,54.0,169
14,block_postal,s3_keys,43081,332728.0,7.723312,3.0,20.0,30.0,57.0,155
0,block_postal_name4,s1_keys,269643,2206821.0,8.184232,1.0,4.0,10.0,92.0,13414


## 9. Estimate Candidate Pair Counts for Each Rule

In [9]:
def estimate_candidate_pairs(s1_path, target_path, rule):
    s1 = s1_path.resolve().as_posix()
    target = target_path.resolve().as_posix()

    query = f"""
    WITH s1_blocks AS (
        SELECT
            "{rule}" AS block_key,
            COUNT(*) AS s1_count
        FROM read_parquet('{s1}')
        WHERE "{rule}" <> ''
        GROUP BY "{rule}"
    ),
    target_blocks AS (
        SELECT
            "{rule}" AS block_key,
            COUNT(*) AS target_count
        FROM read_parquet('{target}')
        WHERE "{rule}" <> ''
        GROUP BY "{rule}"
    )
    SELECT
        COALESCE(
            SUM(s1_count * target_count),
            0
        ) AS candidate_pairs
    FROM s1_blocks
    INNER JOIN target_blocks USING(block_key);
    """
    return int(con.execute(query).fetchone()[0])

candidate_estimates = []
for target_name, target_path in [("S2", S2_KEYS), ("S3", S3_KEYS)]:
    for rule in BLOCK_RULES:
        count = estimate_candidate_pairs(S1_KEYS, target_path, rule)
        candidate_estimates.append({
            "target": target_name,
            "rule": rule,
            "candidate_pairs": count
        })

candidate_estimates_df = pd.DataFrame(candidate_estimates)
candidate_estimates_df["avg_candidates_per_S1"] = (
    candidate_estimates_df["candidate_pairs"] / 2_206_821
)

display(candidate_estimates_df.sort_values("candidate_pairs"))


,target,rule,candidate_pairs,avg_candidates_per_S1
4,S2,block_postal,3347769,1.517010
9,S3,block_postal,3587076,1.625449
2,S2,block_name_exact,35837806,16.239562
7,S3,block_name_exact,39598663,17.943758
3,S2,block_token_pair,241091177,109.248180
8,S3,block_token_pair,253823859,115.017874
1,S2,block_name_prefix6,4338075099,1965.757576
6,S3,block_name_prefix6,4618141585,2092.667047
0,S2,block_postal_name4,6561396691,2973.234662
5,S3,block_postal_name4,7084814599,3210.416522


## 10. Evaluate Empirical Blocking Recall Against Ground Truth

In [10]:
def evaluate_blocking_rule(target_name, target_path, rule):
    gt = GT_PAIRS.resolve().as_posix()
    s1 = S1_KEYS.resolve().as_posix()
    target = target_path.resolve().as_posix()

    query = f"""
    WITH pair_hits AS (
        SELECT
            g.source1_entity_id,
            g.matched_entity_id,
            CASE
                WHEN s1."{rule}" <> '' AND s1."{rule}" = tgt."{rule}" THEN 1
                ELSE 0
            END AS hit
        FROM read_parquet('{gt}') g
        INNER JOIN read_parquet('{s1}') s1
            ON g.source1_entity_id = s1.entity_id
        INNER JOIN read_parquet('{target}') tgt
            ON g.matched_entity_id = tgt.entity_id
        WHERE g.target_source = '{target_name}'
    )
    SELECT
        COUNT(*) AS total_true_pairs,
        SUM(hit) AS recovered_pairs,
        100.0 * SUM(hit) / COUNT(*) AS pair_recall
    FROM pair_hits;
    """
    result = con.execute(query).df()
    result.insert(0, "target", target_name)
    result.insert(1, "rule", rule)
    return result

blocking_results = []
for target_name, target_path in [("S2", S2_KEYS), ("S3", S3_KEYS)]:
    for rule in BLOCK_RULES:
        print(f"Evaluating {rule} -> {target_name}")
        result = evaluate_blocking_rule(target_name, target_path, rule)
        blocking_results.append(result)

blocking_recall_df = pd.concat(blocking_results, ignore_index=True)
display(blocking_recall_df)


Evaluating block_postal_name4 -> S2
Evaluating block_name_prefix6 -> S2
Evaluating block_name_exact -> S2
Evaluating block_token_pair -> S2
Evaluating block_postal -> S2
Evaluating block_postal_name4 -> S3
Evaluating block_name_prefix6 -> S3
Evaluating block_name_exact -> S3
Evaluating block_token_pair -> S3
Evaluating block_postal -> S3


,target,rule,total_true_pairs,recovered_pairs,pair_recall
0,S2,block_postal_name4,3693619,2804617.0,75.931410
1,S2,block_name_prefix6,3693619,2736738.0,74.093673
2,S2,block_name_exact,3693619,1612614.0,43.659457
3,S2,block_token_pair,3693619,2130482.0,57.680069
4,S2,block_postal,3693619,170690.0,4.621213
5,S3,block_postal_name4,3944746,2925557.0,74.163381
6,S3,block_name_prefix6,3944746,2854291.0,72.356775
7,S3,block_name_exact,3944746,1673745.0,42.429728
8,S3,block_token_pair,3944746,2219558.0,56.266183
9,S3,block_postal,3944746,187347.0,4.749279


## 11. Blocking Benchmark: Recall vs. Computational Cost

In [11]:
blocking_report = blocking_recall_df.merge(
    candidate_estimates_df,
    on=["target", "rule"],
    how="left"
)

display(
    blocking_report[
        [
            "target",
            "rule",
            "total_true_pairs",
            "recovered_pairs",
            "pair_recall",
            "candidate_pairs",
            "avg_candidates_per_S1"
        ]
    ].sort_values(["target", "pair_recall"], ascending=[True, False])
)


,target,rule,total_true_pairs,recovered_pairs,pair_recall,candidate_pairs,avg_candidates_per_S1
0,S2,block_postal_name4,3693619,2804617.0,75.931410,6561396691,2973.234662
1,S2,block_name_prefix6,3693619,2736738.0,74.093673,4338075099,1965.757576
3,S2,block_token_pair,3693619,2130482.0,57.680069,241091177,109.248180
2,S2,block_name_exact,3693619,1612614.0,43.659457,35837806,16.239562
4,S2,block_postal,3693619,170690.0,4.621213,3347769,1.517010
5,S3,block_postal_name4,3944746,2925557.0,74.163381,7084814599,3210.416522
6,S3,block_name_prefix6,3944746,2854291.0,72.356775,4618141585,2092.667047
8,S3,block_token_pair,3944746,2219558.0,56.266183,253823859,115.017874
7,S3,block_name_exact,3944746,1673745.0,42.429728,39598663,17.943758
9,S3,block_postal,3944746,187347.0,4.749279,3587076,1.625449


## 12. Evaluate Complete Entity Match Recovery

In [12]:
def evaluate_full_entity_recall(rule):
    gt = GT_PAIRS.resolve().as_posix()
    s1 = S1_KEYS.resolve().as_posix()

    query_parts = []
    for target_name, target_path in [("S2", S2_KEYS), ("S3", S3_KEYS)]:
        target = target_path.resolve().as_posix()
        query_parts.append(f"""
            SELECT
                g.source1_entity_id,
                CASE
                    WHEN s1."{rule}" <> '' AND s1."{rule}" = tgt."{rule}" THEN 1
                    ELSE 0
                END AS hit
            FROM read_parquet('{gt}') g
            INNER JOIN read_parquet('{s1}') s1
                ON g.source1_entity_id = s1.entity_id
            INNER JOIN read_parquet('{target}') tgt
                ON g.matched_entity_id = tgt.entity_id
            WHERE g.target_source = '{target_name}'
        """)

    combined = "\nUNION ALL\n".join(query_parts)

    query = f"""
    WITH pair_results AS (
        {combined}
    ),
    entity_results AS (
        SELECT
            source1_entity_id,
            COUNT(*) AS true_match_count,
            SUM(hit) AS recovered_match_count
        FROM pair_results
        GROUP BY source1_entity_id
    )
    SELECT
        COUNT(*) AS non_singleton_entities,
        SUM(
            CASE
                WHEN recovered_match_count = true_match_count THEN 1
                ELSE 0
            END
        ) AS fully_recovered_entities,
        100.0 * SUM(
            CASE
                WHEN recovered_match_count = true_match_count THEN 1
                ELSE 0
            END
        ) / COUNT(*) AS full_entity_recall
    FROM entity_results;
    """
    result = con.execute(query).df()
    result.insert(0, "rule", rule)
    return result

entity_recall_results = []
for rule in BLOCK_RULES:
    print(f"Full entity recall: {rule}")
    entity_recall_results.append(evaluate_full_entity_recall(rule))

entity_recall_df = pd.concat(entity_recall_results, ignore_index=True)
display(entity_recall_df.sort_values("full_entity_recall", ascending=False))


Full entity recall: block_postal_name4
Full entity recall: block_name_prefix6
Full entity recall: block_name_exact
Full entity recall: block_token_pair
Full entity recall: block_postal


,rule,non_singleton_entities,fully_recovered_entities,full_entity_recall
0,block_postal_name4,2083574,936531.0,44.948296
1,block_name_prefix6,2083574,875850.0,42.035944
3,block_token_pair,2083574,415068.0,19.920963
2,block_name_exact,2083574,214025.0,10.272013
4,block_postal,2083574,57324.0,2.751234


## 13. Save Blocking Benchmark Reports

In [13]:
blocking_report.to_csv(OUTPUT_DIR / "blocking_recall_report.csv", index=False)
block_stats.to_csv(OUTPUT_DIR / "block_size_report.csv", index=False)
candidate_estimates_df.to_csv(OUTPUT_DIR / "candidate_estimates.csv", index=False)
entity_recall_df.to_csv(OUTPUT_DIR / "entity_blocking_recall.csv", index=False)

print("Blocking benchmark outputs successfully saved to:", OUTPUT_DIR.resolve())


Blocking benchmark outputs successfully saved to: D:\ml-model\amazon-ml\outputs


## 14. Analyze Name Uniqueness Partitioned by Country

In [14]:
def unique_names_by_country(path):
    sql_path = path.resolve().as_posix()
    query = f"""
    SELECT
        country,
        COUNT(*) AS records,
        COUNT(DISTINCT business_name) AS unique_business_names,
        COUNT(DISTINCT business_address) AS unique_addresses
    FROM read_csv(
        '{sql_path}',
        delim='\t',
        header=true,
        columns={{
            'entity_id': 'VARCHAR',
            'business_name': 'VARCHAR',
            'business_address': 'VARCHAR',
            'country': 'VARCHAR'
        }}
    )
    GROUP BY country
    ORDER BY country;
    """
    return con.execute(query).df()

print("Source 1 Country Uniqueness:")
display(unique_names_by_country(S1_PATH))

print("Source 2 Country Uniqueness:")
display(unique_names_by_country(S2_PATH))

print("Source 3 Country Uniqueness:")
display(unique_names_by_country(S3_PATH))


Source 1 Country Uniqueness:


,country,records,unique_business_names,unique_addresses
0,India,883188,563618,856280
1,US,1323633,976267,1274326


Source 2 Country Uniqueness:


,country,records,unique_business_names,unique_addresses
0,India,2017799,1685529,1758287
1,US,3016817,2721444,2578974


Source 3 Country Uniqueness:


,country,records,unique_business_names,unique_addresses
0,India,2115547,1832744,1896707
1,US,3170056,2825312,2736057


## 15. Cross-Script Matching Distribution in Ground Truth

In [15]:
def matched_script_analysis(target_name, target_path):
    gt = GT_PAIRS.resolve().as_posix()
    s1 = S1_KEYS.resolve().as_posix()
    target = target_path.resolve().as_posix()

    query = f"""
    SELECT
        s1.name_script AS s1_name_script,
        tgt.name_script AS target_name_script,
        s1.address_script AS s1_address_script,
        tgt.address_script AS target_address_script,
        COUNT(*) AS matched_pairs
    FROM read_parquet('{gt}') g
    INNER JOIN read_parquet('{s1}') s1
        ON g.source1_entity_id = s1.entity_id
    INNER JOIN read_parquet('{target}') tgt
        ON g.matched_entity_id = tgt.entity_id
    WHERE g.target_source = '{target_name}'
    GROUP BY
        s1.name_script,
        tgt.name_script,
        s1.address_script,
        tgt.address_script
    ORDER BY matched_pairs DESC;
    """
    return con.execute(query).df()

script_s2 = matched_script_analysis("S2", S2_KEYS)
script_s3 = matched_script_analysis("S3", S3_KEYS)

print("Top 20 Matched Script Patterns (S1 <-> S2):")
display(script_s2.head(20))

print("Top 20 Matched Script Patterns (S1 <-> S3):")
display(script_s3.head(20))


Top 20 Matched Script Patterns (S1 <-> S2):


,s1_name_script,target_name_script,s1_address_script,target_address_script,matched_pairs
0,LATIN,LATIN,LATIN,LATIN,2921410
1,LATIN,LATIN,LATIN,MIXED,262682
2,LATIN,LATIN,LATIN,EMPTY,164885
3,LATIN,DEVANAGARI,LATIN,LATIN,140488
4,LATIN,DEVANAGARI,LATIN,MIXED,46727
5,LATIN,TELUGU,LATIN,LATIN,21622
6,LATIN,KANNADA,LATIN,LATIN,19585
7,LATIN,TAMIL,LATIN,LATIN,17739
8,LATIN,GUJARATI,LATIN,LATIN,16171
9,LATIN,BENGALI,LATIN,LATIN,15981


Top 20 Matched Script Patterns (S1 <-> S3):


,s1_name_script,target_name_script,s1_address_script,target_address_script,matched_pairs
0,LATIN,LATIN,LATIN,LATIN,3271360
1,LATIN,LATIN,LATIN,MIXED,294770
2,LATIN,LATIN,LATIN,EMPTY,171323
3,LATIN,DEVANAGARI,LATIN,LATIN,76833
4,LATIN,DEVANAGARI,LATIN,MIXED,25746
5,LATIN,MIXED,LATIN,LATIN,19169
6,LATIN,TELUGU,LATIN,LATIN,11754
7,LATIN,KANNADA,LATIN,LATIN,10753
8,LATIN,TAMIL,LATIN,LATIN,9657
9,LATIN,BENGALI,LATIN,LATIN,8891


## 16. Sample Known Ground Truth Pairs for Pairwise Similarity Features

In [17]:
GT_SAMPLE = OUTPUT_DIR / "gt_pairs_sample.parquet"

query = f"""
COPY (
    SELECT *
    FROM read_parquet('{GT_PAIRS.resolve().as_posix()}')
    USING SAMPLE 200000 ROWS
)
TO '{GT_SAMPLE.resolve().as_posix()}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
"""

con.execute(query)
print("Ground truth sample created at:", GT_SAMPLE.resolve())


Ground truth sample created at: D:\ml-model\amazon-ml\outputs\gt_pairs_sample.parquet


## 17. In-Memory Candidate Generation & Ground-Truth Recall Benchmark

This section implements and validates the modular in-memory candidate generation pipeline (`src/blocking.py` and `src/normalization.py`).

### Key Objectives & Mismatch Corrections:
1. **Sampling Alignment**: Naive random slicing of S1, S2, and S3 results in zero true pair overlap because true matches for S1 are dispersed across 5M+ rows in S2/S3. We anchor the evaluation sample to ground truth positive links, ensuring all true matches are retained along with negative distractors.
2. **Feature Engineering & Blocking Keys**: Apply `engineer_features` and `create_block_keys` across multilingual names and addresses.
3. **Inverted Index Candidate Generation**: Index candidate blocks with size limits (`max_block_size=100`) across configured blocking rules.
4. **Integer Index Ground Truth Mapping**: `generate_candidates` outputs zero-based integer row pairs `(s1_idx, target_idx)`. We map ground truth string entity IDs (`S1-...`, `S2-...`, `S3-...`) to corresponding DataFrame row indices (`s1_id_to_idx`, `s2_id_to_idx`, `s3_id_to_idx`) to resolve the type/index mismatch.
5. **Empirical Recall Verification**: Measure true pair recovery percentage and candidate reduction ratio using `calculate_candidate_recall`.


In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from src.normalization import engineer_features
from src.blocking import (
    create_block_keys,
    block_statistics,
    generate_candidates,
    calculate_candidate_recall,
)

# ============================================================
# 1. Load Data (Memory-Safe Sample or Full Dataset)
# ============================================================
# Set SAMPLE_S1_SIZE to an integer (e.g. 10_000) for fast in-memory benchmarking,
# or set to None to run on the entire dataset.
SAMPLE_S1_SIZE = 10_000

if 's1' not in globals() or 's2' not in globals() or 's3' not in globals():
    if SAMPLE_S1_SIZE is not None and SAMPLE_S1_SIZE > 0:
        print(f"Extracting coherent evaluation sample anchored on {SAMPLE_S1_SIZE:,} S1 entities...")
        
        # Extract S1 entities with known positive links from Ground Truth
        if GT_PAIRS.exists():
            s1_sample_ids = con.execute(f"""
                SELECT DISTINCT source1_entity_id
                FROM read_parquet('{GT_PAIRS.resolve().as_posix()}')
                LIMIT {SAMPLE_S1_SIZE}
            """).df()["source1_entity_id"].tolist()
            
            s1_ids_sql = ", ".join(f"'{x}'" for x in s1_sample_ids)
            sample_gt_pairs = con.execute(f"""
                SELECT source1_entity_id, matched_entity_id, target_source
                FROM read_parquet('{GT_PAIRS.resolve().as_posix()}')
                WHERE source1_entity_id IN ({s1_ids_sql})
            """).df()
        else:
            gt_raw = pd.read_csv(GT_PATH, sep="\t", dtype=str, nrows=SAMPLE_S1_SIZE * 3)
            gt_raw = gt_raw.dropna(subset=["matched_entity_ids"])
            s1_sample_ids = gt_raw["source1_entity_id"].iloc[:SAMPLE_S1_SIZE].tolist()
            gt_raw["matched_entity_id"] = gt_raw["matched_entity_ids"].str.split(",")
            sample_gt_pairs = gt_raw[gt_raw["source1_entity_id"].isin(s1_sample_ids)].explode("matched_entity_id").copy()
            sample_gt_pairs["matched_entity_id"] = sample_gt_pairs["matched_entity_id"].str.strip()
            sample_gt_pairs = sample_gt_pairs[sample_gt_pairs["matched_entity_id"] != ""]
            sample_gt_pairs["target_source"] = sample_gt_pairs["matched_entity_id"].apply(
                lambda x: "S2" if str(x).startswith("S2-") else ("S3" if str(x).startswith("S3-") else "UNKNOWN")
            )

        s2_needed_ids = set(sample_gt_pairs[sample_gt_pairs["target_source"] == "S2"]["matched_entity_id"])
        s3_needed_ids = set(sample_gt_pairs[sample_gt_pairs["target_source"] == "S3"]["matched_entity_id"])

        # Query records using DuckDB
        s1 = con.execute(f"""
            SELECT * FROM read_csv('{S1_PATH.resolve().as_posix()}', delim='\t', header=true)
            WHERE entity_id IN (SELECT unnest({s1_sample_ids}))
        """).df()

        # Load S2 true matches + background distractors
        s2_match_df = con.execute(f"""
            SELECT * FROM read_csv('{S2_PATH.resolve().as_posix()}', delim='\t', header=true)
            WHERE entity_id IN (SELECT unnest({list(s2_needed_ids)}))
        """).df()
        s2_distractor_df = con.execute(f"""
            SELECT * FROM read_csv('{S2_PATH.resolve().as_posix()}', delim='\t', header=true)
            USING SAMPLE {SAMPLE_S1_SIZE * 2}
        """).df()
        s2 = pd.concat([s2_match_df, s2_distractor_df], ignore_index=True).drop_duplicates(subset=["entity_id"])

        # Load S3 true matches + background distractors
        s3_match_df = con.execute(f"""
            SELECT * FROM read_csv('{S3_PATH.resolve().as_posix()}', delim='\t', header=true)
            WHERE entity_id IN (SELECT unnest({list(s3_needed_ids)}))
        """).df()
        s3_distractor_df = con.execute(f"""
            SELECT * FROM read_csv('{S3_PATH.resolve().as_posix()}', delim='\t', header=true)
            USING SAMPLE {SAMPLE_S1_SIZE * 2}
        """).df()
        s3 = pd.concat([s3_match_df, s3_distractor_df], ignore_index=True).drop_duplicates(subset=["entity_id"])
    else:
        print("Loading full datasets from TSV files...")
        s1 = pd.read_csv(S1_PATH, sep="\t", dtype=str)
        s2 = pd.read_csv(S2_PATH, sep="\t", dtype=str)
        s3 = pd.read_csv(S3_PATH, sep="\t", dtype=str)

# Ensure consistent 0-based consecutive integer indexing
s1 = s1.reset_index(drop=True)
s2 = s2.reset_index(drop=True)
s3 = s3.reset_index(drop=True)

print(f"Data loaded: S1={len(s1):,} rows, S2={len(s2):,} rows, S3={len(s3):,} rows")

# ============================================================
# 2. Feature Engineering
# ============================================================
print("\n--- Running Feature Engineering ---")
s1_e = engineer_features(s1)
s2_e = engineer_features(s2)
s3_e = engineer_features(s3)

# ============================================================
# 3. Build Blocking Keys
# ============================================================
print("\n--- Creating Blocking Keys ---")
s1_blocks = create_block_keys(s1_e)
s2_blocks = create_block_keys(s2_e)
s3_blocks = create_block_keys(s3_e)

print("\nS1 Block Quality Statistics:")
display(block_statistics(s1_blocks))

print("\nS2 Block Quality Statistics:")
display(block_statistics(s2_blocks))

# ============================================================
# 4. Generate Candidate Pairs
# ============================================================
TEST_RULES = [
    "name_exact",
    "name_core",
    "postal_name6",
    "postal_first",
    "first_last",
    "number_first",
    "number_name4",
]

print("\n--- Generating Candidates (max_block_size=100) ---")
cand_s2 = generate_candidates(s1_e, s2_e, s1_blocks, s2_blocks, TEST_RULES, max_block_size=100)
cand_s3 = generate_candidates(s1_e, s3_e, s1_blocks, s3_blocks, TEST_RULES, max_block_size=100)

print(f"Generated candidate pairs: S1-S2 = {len(cand_s2):,}, S1-S3 = {len(cand_s3):,}")

# ============================================================
# 5. Ground Truth Integer Index Matching Logic
# ============================================================
# Resolves entity_id string vs array integer index mismatch:
# Map each entity_id to its 0-based row index in s1_e, s2_e, s3_e
s1_id_to_idx = {eid: idx for idx, eid in enumerate(s1_e["entity_id"])}
s2_id_to_idx = {eid: idx for idx, eid in enumerate(s2_e["entity_id"])}
s3_id_to_idx = {eid: idx for idx, eid in enumerate(s3_e["entity_id"])}

gt_s2_idx = set()
gt_s3_idx = set()

if GT_PAIRS.exists():
    gt_df = con.execute(f"""
        SELECT source1_entity_id, matched_entity_id, target_source
        FROM read_parquet('{GT_PAIRS.resolve().as_posix()}')
    """).df()
    
    for row in gt_df.itertuples(index=False):
        s1_id = row.source1_entity_id
        target_id = row.matched_entity_id
        tgt = row.target_source
        if s1_id in s1_id_to_idx:
            s1_idx = s1_id_to_idx[s1_id]
            if tgt == "S2" and target_id in s2_id_to_idx:
                gt_s2_idx.add((s1_idx, s2_id_to_idx[target_id]))
            elif tgt == "S3" and target_id in s3_id_to_idx:
                gt_s3_idx.add((s1_idx, s3_id_to_idx[target_id]))
else:
    gt_raw = pd.read_csv(GT_PATH, sep="\t", dtype=str)
    for _, row in gt_raw.iterrows():
        s1_id = row["source1_entity_id"]
        if s1_id not in s1_id_to_idx:
            continue
        s1_idx = s1_id_to_idx[s1_id]
        raw_matches = str(row["matched_entity_ids"]) if pd.notna(row["matched_entity_ids"]) else ""
        if not raw_matches:
            continue
        for m in raw_matches.split(","):
            m = m.strip()
            if m.startswith("S2-") and m in s2_id_to_idx:
                gt_s2_idx.add((s1_idx, s2_id_to_idx[m]))
            elif m.startswith("S3-") and m in s3_id_to_idx:
                gt_s3_idx.add((s1_idx, s3_id_to_idx[m]))

print(f"\nTrue Ground Truth pairs within evaluation scope: S2 = {len(gt_s2_idx):,}, S3 = {len(gt_s3_idx):,}")

# ============================================================
# 6. Measure Recall & Print Final Benchmark
# ============================================================
s2_recall_metrics = calculate_candidate_recall(cand_s2, gt_s2_idx)
s3_recall_metrics = calculate_candidate_recall(cand_s3, gt_s3_idx)

benchmark_summary = pd.DataFrame([
    {"Target": "S2", **s2_recall_metrics},
    {"Target": "S3", **s3_recall_metrics},
])

print("\n================ Candidate Recall Benchmark Report ================")
display(benchmark_summary)


Extracting coherent evaluation sample anchored on 10,000 S1 entities...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded: S1=10,000 rows, S2=37,813 rows, S3=38,914 rows

--- Running Feature Engineering ---
Engineering: 10000 rows
Engineering: 37813 rows
Engineering: 38914 rows

--- Creating Blocking Keys ---

S1 Block Quality Statistics:


,rule,blocks,mean,median,p90,p95,p99,max,blocks_gt_100,blocks_gt_1000
0,name_exact,9779,1.022599,1.0,1.0,1.0,2.0,4,0,0
1,name_core,9633,1.038098,1.0,1.0,1.0,2.0,5,0,0
2,postal_name4,4759,2.101282,1.0,3.0,7.0,20.0,72,0,0
3,postal_name6,6586,1.518372,1.0,2.0,3.0,15.0,72,0,0
4,postal_first,5353,1.868111,1.0,2.0,6.4,19.0,72,0,0
5,first_last,9274,1.078283,1.0,1.0,2.0,3.0,12,0,0
6,number_first,9916,1.008471,1.0,1.0,1.0,1.0,6,0,0
7,number_name4,9906,1.009489,1.0,1.0,1.0,1.0,7,0,0



S2 Block Quality Statistics:


,rule,blocks,mean,median,p90,p95,p99,max,blocks_gt_100,blocks_gt_1000
0,name_exact,35614,1.061745,1.0,1.0,2.0,2.00,7,0,0
1,name_core,33808,1.118463,1.0,1.0,2.0,3.00,8,0,0
2,postal_name4,11672,3.239633,1.0,5.0,10.0,43.00,261,11,0
3,postal_name6,18868,2.004081,1.0,3.0,4.0,22.00,234,1,0
4,postal_first,15082,2.507161,1.0,3.0,5.0,34.19,263,9,0
5,first_last,29292,1.290899,1.0,2.0,2.0,5.00,43,0,0
6,number_first,31380,1.205003,1.0,2.0,2.0,4.00,37,0,0
7,number_name4,30706,1.231453,1.0,2.0,2.0,4.00,34,0,0



--- Generating Candidates (max_block_size=100) ---
0/10,000
0/10,000
Generated candidate pairs: S1-S2 = 186,131, S1-S3 = 200,425


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


True Ground Truth pairs within evaluation scope: S2 = 17,886, S3 = 18,982

================ Candidate Recall Benchmark Report ================


,Target,candidate_pairs,true_pairs,recovered_pairs,pair_recall
0,S2,186131,17886,14576,81.493906
1,S3,200425,18982,15110,79.601728
